# TP4 : Implémentation d'un système RAG (Retrieval-Augmented Generation)

**Étudiant :** ELAALGI Hatim — Module Système Distribué Agentique, Master S3

## Présentation du TP

Ce travaux pratiques (TP) est dédié à la mise en place et à l'évaluation d'une architecture **RAG** (*Retrieval-Augmented Generation*). L'objectif principal est de pallier les limites de connaissances d'un grand modèle de langage (LLM) en lui permettant d'extraire et de s'appuyer sur des informations issues de documents externes spécifiques avant de générer une réponse.

Au cours de ce TP, nous allons aborder les étapes clés suivantes :
* **Le Document Loading & Splitting** : Chargement et découpage des documents en blocs (*chunks*).
* **L'Indexation & Vectorisation** : Transformation des textes en vecteurs et stockage dans une base de données vectorielle.
* **Le Retrieval (Recherche)** : Extraction des segments de texte les plus pertinents vis-à-vis d'une requête utilisateur.
* **La Génération & Évaluation** : Injection du contexte dans le prompt du LLM et analyse de la fidélité de ses réponses.

## Mes choix

| Élément | Notebook du prof | Ce notebook |
|---|---|---|
| Document | `pdfs/nc346.pdf` (note de conjoncture DEPF) | **mon CV** `pdfs/CV_ELAALGI_Hatim_2026_FR_Global.pdf`, comme le propose le prof |
| Découpage | `RecursiveCharacterTextSplitter`, 300 tokens, overlap 20 | identique |
| Embeddings | OpenAI `text-embedding-3-small` | **FastEmbed** local `paraphrase-multilingual-mpnet-base-v2` (768 dimensions, multilingue) |
| Base vectorielle | Chroma, `./store` | identique |
| LLM du RAG | OpenAI `gpt-4o-mini` | Groq `qwen/qwen3.8-27b` |
| LLM juge | OpenAI `gpt-4o` (plus puissant que le LLM du RAG) | Groq `openai/gpt-oss-120b` (120 milliards de paramètres, contre 27 pour le RAG) |

Groq ne fournit pas de modèle d'embedding : les vecteurs sont calculés **localement** dans le conteneur avec FastEmbed (modèles ONNX, sans PyTorch). Comme mon CV est en français, j'ai choisi un modèle **multilingue**. La section 9 le compare à un modèle plus petit.

# 0. Preparation de l'environement 

Dans le terminal `uv init` , ` uv venv `, ` Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope Process ` et ` .venv\Scripts\activate ` .  ` uv add ipykernel ` et  ` uv add langchain ` puis ` uv add langchain-openai ` et  `uv add langchain-text-splitters ` puis  `uv add langchain-community ` finalement pour importer la clee api open ai ` uv add python-dotenv  `

> **Dans mon dépôt :** même environnement uv conteneurisé que pour les TP précédents. J'ai ajouté `uv add langchain-community langchain-text-splitters pypdf chromadb fastembed` (LangChain, `langchain-groq` et `python-dotenv` étaient déjà là depuis le TP3), puis reconstruit l'image. La clé `GROQ_API_KEY` est dans le fichier `.env`, qui n'est pas versionné.

In [ ]:
import json
import os
import shutil
import warnings
warnings.filterwarnings("ignore", message=".*IProgress.*")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import tiktoken
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader
from langchain_community.vectorstores import Chroma
from dotenv import load_dotenv
from IPython.display import Markdown, display

##  LLM

In [ ]:
load_dotenv(override=True)
print("GROQ_API_KEY trouvée :", bool(os.getenv("GROQ_API_KEY")))

In [ ]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0, max_retries=8)

# 1. Loading the pdf, Chunking

Le prof place `nc346.pdf` dans un dossier `pdfs` et propose de choisir son CV à la place. J'ai donc mis mon CV (2 pages) dans `pdfs/`.

In [ ]:
pdf_file = "./pdfs/CV_ELAALGI_Hatim_2026_FR_Global.pdf"

In [ ]:
pdf_loader=PyPDFLoader(pdf_file)

Et si vous voulez charger un dossier qui contient plusieurs fichiers, vous utilisez `PyPDFDirectoryLoader(path="./pdfs")`.

On découpe avec `RecursiveCharacterTextSplitter` et l'encodeur `tiktoken` : des blocs (*chunks*) de 300 tokens au plus, avec un chevauchement (*overlap*) de 20 tokens. Le découpage a besoin de `pypdf` (`uv add pypdf`).

In [ ]:
text_splitter=RecursiveCharacterTextSplitter.from_tiktoken_encoder(encoding_name='o200k_base',chunk_size=300,chunk_overlap=20)

In [ ]:
chunks=pdf_loader.load_and_split(text_splitter)
len(chunks)

In [ ]:
# Ce que contient chaque chunk : page, taille en tokens et début du texte
enc = tiktoken.get_encoding("o200k_base")
pd.set_option("display.max_colwidth", 90)
pd.DataFrame([{"page": c.metadata["page_label"],
               "tokens": len(enc.encode(c.page_content)),
               "début du chunk": c.page_content[:90].replace("\n", " ")} for c in chunks])

@@CHUNKS@@

# 2.vectore Store - ChromaDB et Embedding

*Importer le model d'embedding*

Code du prof (OpenAI, payant) :
```python
from langchain_openai import OpenAIEmbeddings
embedding_model=OpenAIEmbeddings(model='text-embedding-3-small')
```

In [ ]:
from langchain_community.embeddings import FastEmbedEmbeddings
embedding_model = FastEmbedEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2")

v = embedding_model.embed_query("apprentissage fédéré")
print("dimension :", len(v), "| premières valeurs :", [round(x, 3) for x in v[:5]])

Petit test : deux formulations du même sujet doivent avoir des vecteurs plus proches qu'une phrase sans rapport.

In [ ]:
import numpy as np

def cosinus(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

q = embedding_model.embed_query("Quelles langues parle-t-il ?")
for phrase in ["Arabe langue maternelle, français courant, anglais intermédiaire",
               "Pipeline temps réel avec Kafka et FastAPI",
               "La recette du couscous aux sept légumes"]:
    print(f"{cosinus(q, embedding_model.embed_query(phrase)):+.3f}  {phrase}")

Ajouter un dossier "store" et utiliser la commande ` uv add chromadb ` pour ajouter *chromadb*

Je vectorise les chunks et je les stocke dans la base vectorielle **vectorstore**. Je vide `./store` avant : sinon, chaque nouvelle exécution du notebook ajouterait les mêmes chunks en double dans la collection.

In [ ]:
shutil.rmtree("./store", ignore_errors=True)
vectorstore=Chroma.from_documents(chunks,embedding_model,collection_name="cv_elaalgi",persist_directory="./store")
print("vecteurs dans la collection :", vectorstore._collection.count())

# 3.  **Trouver les chunks liée à la questions (Retriever)**

Le code du prof est :
```python
retriever=vectorstore.as_retriever(search_type='similarity',search_Kwargs={'k',5})
```
Le commentaire indique que l'on veut les **5** chunks les plus proches. Mais `search_Kwargs` (avec un K majuscule) n'est pas un paramètre reconnu, et `{'k',5}` est un **ensemble**, pas un dictionnaire. Le paramètre est donc ignoré et le retriever garde sa valeur par défaut. Vérification :

In [ ]:
retriever_prof = vectorstore.as_retriever(search_type='similarity', search_Kwargs={'k',5})
print("code du prof :", len(retriever_prof.invoke("Quels sont ses projets ?")), "chunks renvoyés")

#crée le Retriever : k=5, les cinq chunks les plus proches de la question
retriever=vectorstore.as_retriever(search_type='similarity',search_kwargs={'k':5})
print("corrigé      :", len(retriever.invoke("Quels sont ses projets ?")), "chunks renvoyés")

Test avec la question :
> « Chez quelle entreprise travaille-t-il actuellement ? »

In [ ]:
#test
retrieved_chunks=retriever.invoke("Chez quelle entreprise travaille-t-il actuellement ?")
print("nombre de chunks est :",len(retrieved_chunks))
for d in retrieved_chunks:
    print(f"\n--- page {d.metadata['page_label']} ---\n{d.page_content[:250]} ...")

# 4.  *Prompt Design*

## 4.1 Prompt template

In [ ]:
prompt_template="""
Answer the following question based only on provided context The context is about the CV (resume)
of ELAALGI Hatim, a Master student in Artificial Intelligence & Digital Computing
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer: Je ne sais pas!
<context>
{context}
</context>
<question>
{question}
</question>
"""

### Note sur le contexte
Comme le conseille le prof, j'ai remplacé la description de la note de la DEPF par celle de **mon document** : le CV d'ELAALGI Hatim. Le modèle sait ainsi que « il » dans les questions désigne la personne du CV.

### Gestion des hallucinations
La consigne *« If the answer is not found in the context, answer: Je ne sais pas! »* oblige le modèle à avouer qu'il ne sait pas au lieu d'inventer.

## 4.2 *Retrieving the Relevant Documents*

Je récupère les chunks pertinents, je garde uniquement leur texte (`d.page_content`), puis je les concatène avec le séparateur `". "`.

In [ ]:
user_input="Quel était le sujet de son projet de fin d'études de Licence ?"

relevant_document_chunks=retriever.invoke(user_input)
context_list=[d.page_content for d in relevant_document_chunks]
context_for_query=". ".join(context_list)

print(len(context_for_query), "caractères de contexte")

In [ ]:
len(relevant_document_chunks)

## 4.3 Le prompt

On injecte le contexte et la question dans le template. Le prompt contient :
 1. le message système : ( Answer the following question ............  Je ne sais pas! ) ;
 2. le contexte : entre `<context>` et `</context>` ;
 3. la question : entre `<question>` et `</question>`.

In [ ]:
prompt=prompt_template.format(context=context_for_query,question=user_input)
print(prompt[:900], "\n[...]\n", prompt[-250:])

# 5. *Récupérer la réponse de la part de llm*

In [ ]:
resp=llm.invoke(prompt)
display(Markdown(resp.content))

## 6. Regroupement dans une application RAG

### Définition de la fonction RAG
`RAG(query, llm, prompt_template)` enchaîne les trois étapes :
1. **le retriever** cherche les chunks proches de `query` et garde leur texte ;
2. **la concaténation** de ces textes forme le contexte ;
3. **la génération** : le contexte et la question sont injectés dans le template, puis on appelle `llm.invoke()`.

In [ ]:
def RAG(query,llm=llm,prompt_template=prompt_template):
    context_docs=retriever.invoke(query)
    context_list=[d.page_content for d in context_docs]
    context_for_query=". ".join(context_list)
    prompt=prompt_template.format(context=context_for_query, question=query)
    resp=llm.invoke(prompt)
    return resp.content

### Appel de RAG
D'abord la question du prof, qui n'a **aucun rapport** avec mon CV :

In [ ]:
resp=RAG(" qu'ils sont les événements culturels dans le sud-est de Maroc ? ")
display(Markdown(resp))

Puis une question dont la réponse est dans le CV :

In [ ]:
resp=RAG("Quels résultats obtient-il avec le VAE dans son projet de détection de fraude ?")
display(Markdown(resp))

# 7. Évaluation de la réponse du LLM

## 7.1 Architecture

![Architecture RAG + LLM juge](./rag_juge.gif)

Le RAG (LLM1) génère une réponse. Le **LLM juge** reçoit la question, le contexte retrouvé et cette réponse, puis renvoie un **score**.

In [ ]:
#question ou query
user_input="Chez quelle entreprise travaille-t-il actuellement et sur quel projet ?"

Récupérer le context 

In [ ]:
relevant_document_chunks=retriever.invoke(user_input)
context_list=[d.page_content for d in relevant_document_chunks]
context_for_query=". ".join(context_list)

In [ ]:
user_message_template="""
###Question
{question}
###Context
{context}
###Answer
{answer}
"""

In [ ]:
# Réponse d'un RAG
answer=RAG(user_input)
display(Markdown(answer))

## 7.2- *Groundness*

Le prompt pour le juge 

In [ ]:
groundness_rater_system_message="""
Vous êtes chargé d'évaluer des réponses générées par une IA à des questions posées par des utilisateurs.
On vous présentera une question, le contexte utilisé par le système d'IA pour générer la réponse, ainsi qu'une réponse générée par l'IA à la question.
Dans l'entrée, la question commencera par ###Question, le contexte commencera par ###Context, et la réponse générée par L'IA commencera par ###Answer.
Critères d'évaluation :
La tâche consiste à juger dans quelle mesure la réponse respecte la métrique.

1- La métrique n'est pas respectée du tout
2- La métrique n'est respectée que dans une mesure limitée
3- La métrique est respectée dans une bonne mesure
4- La métrique est respectée en grande partie
5- La métrique est entièrement respectée

Métrique :

La réponse doit être dérivée uniquement des informations présentées dans le contexte.

Instructions:

Écrivez d'abord les étapes nécessaires pour évaluer la réponse selon la métrique.
Donnez une explication étape par étape indiquant si la réponse respecte la métrique, en considérant la question et le contexte comme entrées.
Évaluez ensuite dans quelle mesure la métrique est respectée.
Utilisez les informations précédentes pour noter la réponse selon les critères d'évaluation et attribuer un score.
"""

In [ ]:
# Concernant le juge, il est nécessaire d'employer un modèle plus puissant que celui mis en œuvre par le RAG.
groundness_checker=ChatGroq(
      model="openai/gpt-oss-120b",
      temperature=0,
      max_retries=8
)

In [ ]:
def evaluate(system_message, user_message_template,question, model=groundness_checker):
    retrieved_chunks=retriever.invoke(question)
    context_list=[d.page_content for d in retrieved_chunks]
    context=". ".join(context_list)
    answer=RAG(question)
    prompt=f"""
     { system_message}\n
     USER:
     {user_message_template.format(question=question, context=context, answer=answer)}
     """
    juge_response=model.invoke(prompt)
    return juge_response.content

In [ ]:
#évaluer
resp=evaluate(groundness_rater_system_message, user_message_template, user_input)
display(Markdown(resp))

### Échelle de notation pour la fidélité au contexte (Faithfulness)

| Score | Signification | Description |
| :--- | :--- | :--- |
| **5 / 5** | **Excellent (Parfait)** | La réponse utilise **uniquement** les informations du contexte. Aucune hallucination, aucune extrapolation. |
| **4 / 5** | **Bon** | La réponse est correcte et s'appuie sur le contexte, mais la formulation pourrait être légèrement plus précise. |
| **3 / 5** | **Moyen** | La réponse est partiellement vraie, mais elle omet un détail important du contexte ou manque de clarté. |
| **2 / 5** | **Médiocre** | La réponse contient des extrapolations ou des informations non vérifiables avec le contexte fourni. |
| **1 / 5** | **Mauvais (Hallucination)** | La réponse est totalement hors-sujet, fausse, ou invente des données qui ne sont pas du tout dans le contexte. |

## 8. Évaluation sur un jeu de questions

Une seule question ne suffit pas pour juger un RAG. J'ai donc écrit **9 questions** sur mon CV et, pour chacune, la réponse attendue (que je connais, puisque c'est mon CV) :
- 7 questions dont la réponse **est dans le CV**, sur des parties différentes (formation, expérience, projets, langues) ;
- 1 question **absente** du CV (date de naissance) ;
- 1 question **hors sujet** (la question du prof sur la note de conjoncture).

Pour chaque question, je mesure :
1. **retrieval** : le chunk qui contient la réponse est-il parmi les 5 chunks retrouvés ? On teste la présence d'un mot-clé de la réponse attendue dans le contexte ;
2. **exactitude** : est-ce que je juge la réponse correcte ? (le juge ne vérifie que la fidélité au contexte, pas la justesse) ;
3. **fidélité** : la note du juge. Pour pouvoir l'extraire automatiquement, j'ajoute au prompt du juge une consigne de format pour la dernière ligne.

In [ ]:
import re

questions = [
    # (question, mot-clé qui doit être dans le contexte, réponse attendue)
    ("Dans quel master est-il inscrit et quand obtiendra-t-il son diplôme ?", "Digital Computing",
     "Master IA & Digital Computing, FST Université Moulay Slimane, diplôme prévu en juin 2027"),
    ("Chez quelle entreprise travaille-t-il actuellement et sur quel projet ?", "Synertic",
     "Synertic (ESN), copilote d'investigation de sécurité à base d'agents LLM"),
    ("Quel était le sujet de son projet de fin d'études de Licence ?", "ATLAS",
     "Chatbot ATLAS au HCP : NL vers SQL sur les bases statistiques, Flask, Gemini"),
    ("Quels résultats obtient-il avec le VAE dans son projet de détection de fraude ?", "0,1139",
     "AUC-PR 0,1139 contre 0,0497 (+129 %), rappel 44,2 %, coût réduit de 24 %"),
    ("Combien de clients participent à l'apprentissage fédéré du projet NIDS-FL ?", "FedAvg",
     "3 clients (FedAvg)"),
    ("Quelles langues parle-t-il et à quel niveau ?", "maternelle",
     "arabe (maternelle), français (courant), anglais (intermédiaire)"),
    ("Quelle technologie utilise-t-il pour l'avatar 3D du projet Ibn Battuta ?", "Three.js",
     "Three.js (avec FastAPI, React 19, Qwen3-VL-4B)"),
    ("Quelle est sa date de naissance ?", None, "absente du CV : Je ne sais pas"),
    ("QU'il est la capitalisation boursière de Maroc pour 2025 ?", None, "hors sujet : Je ne sais pas"),
]

format_score = "\nTerminez votre réponse par une dernière ligne au format exact : Score : X/5"


def extraire_score(jugement):
    notes = re.findall(r"Score\W{0,6}([1-5])", jugement, flags=re.IGNORECASE)
    return int(notes[-1]) if notes else None


lignes = []
for question, mot_cle, attendu in questions:
    contexte = ". ".join(d.page_content for d in retriever.invoke(question))
    reponse = RAG(question)
    jugement = evaluate(groundness_rater_system_message + format_score, user_message_template, question)
    lignes.append({
        "question": question,
        "chunk utile retrouvé": "—" if mot_cle is None else ("oui" if mot_cle in contexte else "NON"),
        "réponse du RAG": reponse.strip(),
        "réponse attendue": attendu,
        "fidélité (juge)": extraire_score(jugement),
    })

resultats = pd.DataFrame(lignes)
pd.set_option("display.max_colwidth", 220)
resultats

@@EVAL@@

## 9. Influence du modèle d'embedding sur le retrieval

Le retrieval ne dépend que du modèle d'embedding, pas du LLM. Je compare donc mon modèle (`mpnet`, 768 dimensions, environ 1 Go) à un modèle multilingue plus léger de la même famille (`MiniLM-L12`, 384 dimensions, environ 0,2 Go). Pour chaque question à réponse connue, je classe **tous** les chunks du CV par similarité et je note le **rang** du premier chunk qui contient le mot-clé. Si ce rang dépasse 5, le LLM ne voit jamais l'information. Cette expérience n'appelle aucun LLM.

In [ ]:
def rangs(store):
    n = store._collection.count()
    out = []
    for question, mot_cle, _ in questions:
        if mot_cle is None:
            continue
        docs = store.similarity_search(question, k=n)
        out.append(next((r for r, d in enumerate(docs, start=1) if mot_cle in d.page_content), None))
    return out


emb_minilm = FastEmbedEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
store_minilm = Chroma.from_documents(chunks, emb_minilm, collection_name="cv_elaalgi_minilm", persist_directory="./store")

comparaison = pd.DataFrame({
    "question": [q for q, m, _ in questions if m is not None],
    "rang (mpnet, 768 d)": rangs(vectorstore),
    "rang (MiniLM, 384 d)": rangs(store_minilm),
})
comparaison

In [ ]:
for col in ["rang (mpnet, 768 d)", "rang (MiniLM, 384 d)"]:
    r = comparaison[col]
    print(f"{col:<22} dans le top 5 : {(r <= 5).sum()}/{len(r)} | rang moyen : {r.mean():.1f} | MRR : {(1 / r).mean():.2f}")

@@EMB@@

@@CONCLUSION@@